# 04. Real / Imaginary Split MP-PCA — 6-Core Parallel

`02_Add_noise_Open_QSM_FULL_VOLUME_REVISED.ipynb`에서 생성한
**5-echo noisy complex GRE**를 Real과 Imaginary 성분으로 분리한 뒤,
각 성분에 MP-PCA를 독립적으로 적용한다.

이번 버전은 서버 CPU 6개를 실제로 활용하기 위해
**Noise 10%, 30%, 50% × Real/Imaginary = 총 6개 작업을 동시에 처리**한다.

병렬 구조:

```text
Process 1 → 10% Real
Process 2 → 10% Imag
Process 3 → 30% Real
Process 4 → 30% Imag
Process 5 → 50% Real
Process 6 → 50% Imag
```

6개 작업이 모두 끝난 뒤 같은 noise level의 Real과 Imaginary를 다시 결합한다.

```text
10% : Real + j·Imag
30% : Real + j·Imag
50% : Real + j·Imag
```

실험 조건은 기존 버전과 동일하다.

- 입력: `(X, Y, Z, 5)` noisy complex GRE
- Real / Imaginary 각각 `(X, Y, Z, 5)`
- Real과 Imaginary는 서로 독립적으로 MP-PCA 수행
- **Brain mask는 denoising에 사용하지 않음**
- **전체 volume denoising**
- Brain mask는 최종 평가 ROI용으로만 저장
- Patch radius = 3 → `7 × 7 × 7`
- Real/Imaginary의 음수 값을 보존하기 위해 `localpca_dn.py`의 `mppca()` 사용


In [ ]:
import os
import time
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from pathlib import Path
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed


### 환경설정

입력 noisy data:

`./data/noisy_data/`

- `qsm_siemens_monopolar_noisy_10.npz`
- `qsm_siemens_monopolar_noisy_30.npz`
- `qsm_siemens_monopolar_noisy_50.npz`

출력:

`./data/denoised/real_imag/`

서버 실행 시 이 notebook과 함께 `localpca_dn.py`가
`~/workspace`에서 import 가능해야 한다.

6-core 병렬화는 noise level × component 단위로 수행한다.

```text
10% Real
10% Imag
30% Real
30% Imag
50% Real
50% Imag
```

각 worker는 자기 성분의 결과를 임시 `.npy` 파일로 저장한다.
6개 worker가 모두 끝난 뒤 부모 프로세스가 같은 noise level의
Real/Imag를 결합하여 최종 `.mat` 파일을 만든다.


In [ ]:
NOISE_LEVELS = [10, 30, 50]
COMPONENTS = ["real", "imag"]

PATCH_R = 3
N_WORKERS = 6

NOISY_DIR = Path("./data/noisy_data").resolve()
OUT_DIR = Path("./data/denoised/real_imag").resolve()
TMP_DIR = (OUT_DIR / "_parallel_tmp").resolve()

OUT_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

print("Noisy directory :", NOISY_DIR)
print("Output directory:", OUT_DIR)
print("Temp directory  :", TMP_DIR)
print("Patch radius    :", PATCH_R)
print("Parallel workers:", N_WORKERS)
print(
    "Patch size      :",
    f"{2 * PATCH_R + 1} x {2 * PATCH_R + 1} x {2 * PATCH_R + 1}"
)


### 1. 입력 파일 확인

부모 프로세스에서는 세 noise level의 파일 존재 여부만 확인한다.

큰 4D 배열을 부모 프로세스에서 모두 읽지 않고,
각 worker가 담당 noise level의 `.npz`를 직접 읽도록 하여
프로세스 간 데이터 복사와 메모리 사용을 줄인다.


In [ ]:
input_paths = {}

for noise_level in NOISE_LEVELS:
    noisy_path = (
        NOISY_DIR /
        f"qsm_siemens_monopolar_noisy_{noise_level}.npz"
    )

    if not noisy_path.exists():
        raise FileNotFoundError(
            f"Noise 파일을 찾을 수 없습니다: {noisy_path}\n"
            "먼저 02_Add_noise_Open_QSM_FULL_VOLUME_REVISED.ipynb를 실행하세요."
        )

    input_paths[noise_level] = noisy_path
    print(f"{noise_level}% 입력 확인 -> {noisy_path}")

print("\n모든 입력 파일 확인 완료")


### 2. Component 단위 MP-PCA worker

각 worker는 `(noise level, component)` 하나만 담당한다.

예:

```text
(10%, real)
(10%, imag)
(30%, real)
...
```

Worker 처리 순서:

1. 담당 noise level의 noisy complex GRE 로드
2. Real 또는 Imaginary 성분 하나만 추출
3. 전체 volume MP-PCA 수행
4. 결과를 임시 `.npy` 파일로 저장
5. 완료 정보만 부모 프로세스로 반환

Real/Imaginary의 음수 성분을 보존하기 위해
`localpca_dn.py`의 `mppca()`를 사용한다.

`mask_brain`은 worker MP-PCA에 전달하지 않는다.


In [ ]:
def run_component_worker(
    noise_level,
    component,
    noisy_path_str,
    tmp_dir_str,
    patch_r
):
    import os
    import time
    import numpy as np
    from pathlib import Path

    from localpca_dn import mppca

    start_time = time.time()

    noisy_path = Path(noisy_path_str)
    tmp_dir = Path(tmp_dir_str)
    tmp_dir.mkdir(parents=True, exist_ok=True)

    component = component.lower()

    if component not in ("real", "imag"):
        raise ValueError(f"Unknown component: {component}")

    print(
        f"[PID {os.getpid()}] START | "
        f"Noise {noise_level}% | {component.upper()}",
        flush=True
    )

    data = np.load(noisy_path, allow_pickle=False)
    noisy_complex = data["noisy_complex"].astype(np.complex64)
    mask = data["mask_brain"].astype(bool)

    if noisy_complex.ndim != 4:
        raise ValueError(
            f"{noise_level}% noisy_complex는 4D여야 합니다. "
            f"현재 shape={noisy_complex.shape}"
        )

    if noisy_complex.shape[-1] != 5:
        raise ValueError(
            f"{noise_level}% 데이터는 5 echoes를 예상합니다. "
            f"실제={noisy_complex.shape[-1]}"
        )

    if mask.shape != noisy_complex.shape[:3]:
        raise ValueError(
            f"{noise_level}% mask shape {mask.shape} != "
            f"spatial shape {noisy_complex.shape[:3]}"
        )

    if component == "real":
        noisy_component = noisy_complex.real.astype(np.float32)
    else:
        noisy_component = noisy_complex.imag.astype(np.float32)

    del noisy_complex

    print(
        f"[PID {os.getpid()}] MP-PCA | "
        f"Noise {noise_level}% | {component.upper()} | "
        f"shape={noisy_component.shape} | patch_radius={patch_r} | "
        "mask=NOT_USED | ROI=FULL_VOLUME",
        flush=True
    )

    deno_component = mppca(
        noisy_component,
        patch_radius=patch_r
    ).astype(np.float32)

    if deno_component.shape != noisy_component.shape:
        raise ValueError(
            f"{noise_level}% {component} denoised shape "
            f"{deno_component.shape} != input {noisy_component.shape}"
        )

    tmp_path = (
        tmp_dir /
        f"{noise_level}_{component}_r{patch_r}.npy"
    )

    np.save(tmp_path, deno_component)

    elapsed = time.time() - start_time

    print(
        f"[PID {os.getpid()}] DONE | "
        f"Noise {noise_level}% | {component.upper()} | "
        f"{elapsed / 60:.2f} min | {tmp_path.name}",
        flush=True
    )

    return {
        "noise_level": int(noise_level),
        "component": component,
        "tmp_path": str(tmp_path),
        "elapsed_sec": float(elapsed),
        "pid": int(os.getpid()),
    }


### 3. 총 6개 작업을 6코어에서 동시에 실행

Linux 서버에서 `fork` multiprocessing context를 사용한다.

```text
Worker 1 → 10% Real
Worker 2 → 10% Imag
Worker 3 → 30% Real
Worker 4 → 30% Imag
Worker 5 → 50% Real
Worker 6 → 50% Imag
```

서버 제출 조건:

- `-c 6`
- `OMP_NUM_THREADS=1`
- `OPENBLAS_NUM_THREADS=1`
- `MKL_NUM_THREADS=1`

따라서 6개 worker가 각각 한 CPU core를 중심으로 사용하게 된다.

주의: CPU 6코어를 허용하는 계정/프로파일에서 실행해야 한다.


In [ ]:
parallel_start = time.time()
worker_results = {}

ctx = mp.get_context("fork")

tasks = [
    (noise_level, component)
    for noise_level in NOISE_LEVELS
    for component in COMPONENTS
]

print("=" * 70)
print("6-Core Parallel Real / Imag Split MP-PCA 시작")
print(f"Tasks        : {tasks}")
print(f"Workers      : {N_WORKERS}")
print(f"Patch radius : {PATCH_R}")
print("=" * 70)

with ProcessPoolExecutor(
    max_workers=N_WORKERS,
    mp_context=ctx
) as executor:

    future_to_task = {
        executor.submit(
            run_component_worker,
            noise_level,
            component,
            str(input_paths[noise_level]),
            str(TMP_DIR),
            PATCH_R
        ): (noise_level, component)
        for noise_level, component in tasks
    }

    for future in as_completed(future_to_task):
        noise_level, component = future_to_task[future]

        try:
            result = future.result()
            worker_results[(noise_level, component)] = result

            print(
                f"MAIN: {noise_level}% {component.upper()} 완료 | "
                f"PID={result['pid']} | "
                f"{result['elapsed_sec'] / 60:.2f} min"
            )

        except Exception as e:
            print(
                f"MAIN: {noise_level}% {component.upper()} FAILED "
                f"-> {repr(e)}"
            )
            raise

parallel_elapsed = time.time() - parallel_start

print("\n" + "=" * 70)
print("6개 component MP-PCA 병렬 처리 완료")
print(f"병렬 MP-PCA wall time: {parallel_elapsed / 60:.2f} min")
print("=" * 70)


### 4. Real / Imaginary 재결합 및 최종 결과 저장

6개의 component worker가 모두 끝난 뒤,
같은 noise level의 Real과 Imaginary 결과를 불러와 재결합한다.

```text
deno_complex = deno_real + j × deno_imag
```

최종 출력:

- `denoised_real_imag_10_r3.mat`
- `denoised_real_imag_30_r3.mat`
- `denoised_real_imag_50_r3.mat`

Brain mask와 noise metadata는 원래 noisy `.npz`에서 다시 읽어 저장한다.


In [ ]:
output_paths = {}

combine_start = time.time()

for noise_level in NOISE_LEVELS:
    real_tmp = TMP_DIR / f"{noise_level}_real_r{PATCH_R}.npy"
    imag_tmp = TMP_DIR / f"{noise_level}_imag_r{PATCH_R}.npy"

    if not real_tmp.exists():
        raise FileNotFoundError(real_tmp)
    if not imag_tmp.exists():
        raise FileNotFoundError(imag_tmp)

    deno_real = np.load(real_tmp).astype(np.float32)
    deno_imag = np.load(imag_tmp).astype(np.float32)

    if deno_real.shape != deno_imag.shape:
        raise ValueError(
            f"{noise_level}% Real/Imag shape mismatch: "
            f"{deno_real.shape} != {deno_imag.shape}"
        )

    deno_complex = (
        deno_real.astype(np.complex64)
        + 1j * deno_imag.astype(np.complex64)
    ).astype(np.complex64)

    noisy_npz = np.load(
        input_paths[noise_level],
        allow_pickle=False
    )

    output_path = (
        OUT_DIR /
        f"denoised_real_imag_{noise_level}_r{PATCH_R}.mat"
    )

    save_data = {
        "deno_real": deno_real,
        "deno_imag": deno_imag,
        "deno_complex": deno_complex,
        "mask_brain": noisy_npz["mask_brain"].astype(np.uint8),
        "noise_level": np.int32(noise_level),
        "noise_ratio": np.float32(noisy_npz["noise_ratio"]),
        "sigma": np.float32(noisy_npz["sigma"]),
        "seed": np.int32(noisy_npz["seed"]),
        "overall_mean": np.float32(noisy_npz["overall_mean"]),
        "patch_radius": np.int32(PATCH_R),
        "echo_times": (
            noisy_npz["echo_times"].astype(np.float64)
            if "echo_times" in noisy_npz.files
            else np.full(5, np.nan, dtype=np.float64)
        ),
        "field_strength": np.float32(
            noisy_npz["field_strength"]
            if "field_strength" in noisy_npz.files
            else np.nan
        ),
        "denoise_region": "full_volume",
        "mask_usage": "evaluation_only",
        "parallel_workers": np.int32(N_WORKERS),
        "parallel_tasks": "10/30/50 x real/imag",
        "method": "real_imag_split_mppca",
    }

    sio.savemat(
        output_path,
        save_data,
        do_compression=True
    )

    output_paths[noise_level] = output_path

    size_mb = output_path.stat().st_size / (1024 ** 2)

    print(
        f"{noise_level}% 재결합 및 저장 완료 | "
        f"{output_path.name} | {size_mb:.1f} MB"
    )

    del deno_real, deno_imag, deno_complex, noisy_npz

combine_elapsed = time.time() - combine_start

print(
    f"재결합/저장 시간: {combine_elapsed / 60:.2f} min"
)


### 5. 임시 파일 정리 및 결과 시각화

최종 `.mat` 저장이 완료된 뒤,
6개 worker가 만든 임시 `.npy` 파일을 삭제한다.

그 후 noise level 하나씩 입력과 최종 결과를 다시 읽어 확인한다.

표시 항목:

- Noisy Magnitude
- Denoised Magnitude
- Removed = `Noisy - Denoised`

`Removed`는 최종 error map이 아니라,
Real/Imag Split MP-PCA가 입력에서 제거한 성분을 확인하기 위한 시각화이다.


In [ ]:
# 임시 component 결과 삭제
for noise_level in NOISE_LEVELS:
    for component in COMPONENTS:
        tmp_path = (
            TMP_DIR /
            f"{noise_level}_{component}_r{PATCH_R}.npy"
        )

        if tmp_path.exists():
            tmp_path.unlink()

try:
    TMP_DIR.rmdir()
except OSError:
    pass

print("임시 병렬 파일 정리 완료")

echo_idx = 0

for noise_level in NOISE_LEVELS:
    noisy_npz = np.load(
        input_paths[noise_level],
        allow_pickle=False
    )
    noisy_complex = noisy_npz["noisy_complex"].astype(np.complex64)

    mat = sio.loadmat(output_paths[noise_level])
    deno_complex = mat["deno_complex"].astype(np.complex64)

    if deno_complex.shape != noisy_complex.shape:
        raise ValueError(
            f"{noise_level}% visualization shape mismatch: "
            f"{deno_complex.shape} != {noisy_complex.shape}"
        )

    noisy_mag = np.abs(noisy_complex).astype(np.float32)
    deno_mag = np.abs(deno_complex).astype(np.float32)

    slice_idx = noisy_mag.shape[2] // 2

    noisy_2d = noisy_mag[:, :, slice_idx, echo_idx]
    deno_2d = deno_mag[:, :, slice_idx, echo_idx]
    removed_2d = noisy_2d - deno_2d

    vmin, vmax = np.percentile(noisy_2d, (1, 99))

    dv = float(np.percentile(np.abs(removed_2d), 99))
    dv = max(dv, 1e-8)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))

    ax[0].imshow(
        noisy_2d,
        cmap="gray",
        vmin=vmin,
        vmax=vmax
    )
    ax[0].set_title(f"Noisy ({noise_level}%)")

    ax[1].imshow(
        deno_2d,
        cmap="gray",
        vmin=vmin,
        vmax=vmax
    )
    ax[1].set_title("Real/Imag Split MP-PCA")

    ax[2].imshow(
        removed_2d,
        cmap="bwr",
        vmin=-dv,
        vmax=dv
    )
    ax[2].set_title("Removed (Noisy - Denoised)")

    for a in ax:
        a.axis("off")

    fig.suptitle(
        f"6-Core Parallel Real/Imag Split MP-PCA | "
        f"Noise {noise_level}% | Echo {echo_idx + 1} | Slice {slice_idx}",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()

    del noisy_complex, deno_complex, noisy_mag, deno_mag, mat


### 완료

6개 독립 MP-PCA 작업:

```text
10% Real / 10% Imag
30% Real / 30% Imag
50% Real / 50% Imag
```

을 6개 worker로 동시에 수행한 뒤
noise level별 complex GRE로 재결합하여
`./data/denoised/real_imag/`에 저장하면 완료이다.


In [ ]:
print("04_Real_Imag_6 완료")
print("Output directory:", OUT_DIR)
print("Parallel workers:", N_WORKERS)
